# Visualize Raw DNS Snapshot Timeseries (with Interpolation)

This notebook loads a sequence of unprocessed 3D snapshots from raw binary files, combines them into a 4D timeseries, interpolates 2D slices onto uniform grids, and visualizes them.

**Workflow:**
1.  Set all parameters in the **User Configuration** cell.
2.  Run the **Load Snapshot Timeseries** cell once to load all binary files into memory.
3.  Run the **Calculate Global Color Scale** cell to determine the color limits for all plots.
4.  Modify slice indices in the parameter cells for each plot type and run the individual plot cells as needed.

### 1. Setup and Imports

In [ ]:
import numpy as np
from pathlib import Path
import logging
from tqdm.notebook import tqdm

# Import all necessary plotting functions from the refactored package
from mhd_surrogate_core.plotting import (
    plot_interpolated_xz_slice,
    plot_interpolated_xy_slice,
    plot_interpolated_yz_slice,
    plot_interpolated_z_time_evolution,
    plot_interpolated_y_time_evolution,
    plot_interpolated_x_time_evolution
)

### 2. User Configuration

**Action Required:** Set your parameters in this cell. You can re-run this cell to apply changes without reloading the data.

In [ ]:
# <<< 1. SET SNAPSHOT FILE PATH AND PARAMETERS >>>
snapshot_dir = Path("/raid/skowronek/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/raw/")
file_prefix = "patt3d_vx3d_"
# Specify the range of time indices for the files you want to load
time_indices = range(609, 617)

nx, ny, nz = 2301, 481, 121  # Grid dimensions for a single snapshot
source_channel_labels = ['vx', 'vy', 'vz', 'T'] # IMPORTANT: Must be in file order

# <<< 2. DEFINE CHANNEL NAME MAPPING >>>
channel_map = {
    'vx': 'u',
    'vy': 'v',
    'vz': 'w'
}

# <<< 3. SET INTERPOLATION GRID SIZE >>>
num_interp_points_y = 1024
num_interp_points_z = 1024

# <<< 4. SET GLOBAL PLOT SIZING AND UNIT PARAMETERS >>>
global_base_size = 20
global_min_size = 2
global_unit_label = "m/s"

# <<< 5. SET GLOBAL COLOR SCALE OVERRIDE (OPTIONAL) >>>
global_vmin_override = None
global_vmax_override = None

print("Configuration set. You can now run the 'Load Snapshot Timeseries' cell.")

### 3. Load Snapshot Timeseries

**Run this cell only once** to load the sequence of raw binary files into a 4D array in memory.

In [ ]:
# --- Data Loading and Initialization ---
timeseries_data = None
raw_coords = {}

snapshot_files = [snapshot_dir / f"{file_prefix}{i:06d}" for i in time_indices]
if not all(f.exists() for f in snapshot_files):
    logging.error(f"ERROR: Not all snapshot files were found in {snapshot_dir}. Please check the path, prefix, and time indices.")
else:
    all_snapshots = []
    try:
        # Load coordinates from the first file only
        first_file = snapshot_files[0]
        input_dtype = np.float64
        with open(first_file, 'rb') as f:
            raw_coords['x'] = np.fromfile(f, dtype=input_dtype, count=nx)
            raw_coords['y'] = np.fromfile(f, dtype=input_dtype, count=ny)
            raw_coords['z'] = np.fromfile(f, dtype=input_dtype, count=nz)
        raw_coords['labels'] = source_channel_labels
        
        # Load all snapshots in the sequence
        for f_path in tqdm(snapshot_files, desc="Loading snapshots"):
            with open(f_path, 'rb') as f:
                # Skip coordinates
                f.seek((nx + ny + nz) * np.dtype(input_dtype).itemsize)
                channel_data_1d = np.fromfile(f, dtype=input_dtype)
            
            num_input_channels = len(source_channel_labels)
            data_4d_physical = channel_data_1d.reshape((nz, num_input_channels, ny, nx))
            snapshot_3d = data_4d_physical.transpose(3, 2, 0, 1).astype(np.float32)
            all_snapshots.append(snapshot_3d)
        
        # Stack into a single 4D array (5D including channels)
        timeseries_data = np.stack(all_snapshots, axis=0)
        
        print("Timeseries data loaded successfully into memory.")
        print(f"Final data shape: {timeseries_data.shape} (time, x, y, z, channel)")
    except Exception as e:
        logging.error(f"Failed to load or process snapshot files: {e}")
        logging.error("Please check grid dimensions (nx, ny, nz) and source channel labels.")

# Identify velocity components to be plotted
velocity_components = sorted([c for c in raw_coords.get('labels', []) if c in channel_map])
if not velocity_components:
    logging.warning("Warning: No velocity components matching the channel_map found.")

### 4. Calculate Global Color Scale

Run this cell to determine the color scale that will be applied to **all** subsequent plots.

In [ ]:
# --- Calculate the global vmin and vmax across all velocity components ---
global_vmin, global_vmax = None, None

if global_vmin_override is not None and global_vmax_override is not None:
    global_vmin = global_vmin_override
    global_vmax = global_vmax_override
    print(f"Using user-defined global color scale: [{global_vmin:.3f}, {global_vmax:.3f}]")
elif timeseries_data is not None and velocity_components:
    all_vc_data = []
    for vc in velocity_components:
        vc_idx = raw_coords['labels'].index(vc)
        all_vc_data.append(timeseries_data[..., vc_idx])
    
    global_vmin = min(d.min() for d in all_vc_data)
    global_vmax = max(d.max() for d in all_vc_data)
    print(f"Auto-calculated global color scale for all plots: [{global_vmin:.3f}, {global_vmax:.3f}]")

---

### 5. Spatial Slice Plots (2D)

#### 5.1 X-Z Slices (at constant Y)

In [ ]:
# === Parameters for X-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_time_index_xz = 0 # Select time index from the loaded sequence
plot_y_index_xz = ny // 2
# ---
print(f"Using time-index {plot_time_index_xz} and y-index {plot_y_index_xz} for X-Z slice plots.")

In [ ]:
# === Plot X-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xz],
        raw_coords=raw_coords,
        channel='vx',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xz],
        raw_coords=raw_coords,
        channel='vy',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xz],
        raw_coords=raw_coords,
        channel='vz',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.2 X-Y Slices (at constant Z)

In [ ]:
# === Parameters for X-Y Slices ===
# <<< MODIFY THESE VALUES >>>
plot_time_index_xy = 0
plot_z_index_xy = nz // 2
# ---
print(f"Using time-index {plot_time_index_xy} and z-index {plot_z_index_xy} for X-Y slice plots.")

In [ ]:
# === Plot X-Y Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xy],
        raw_coords=raw_coords,
        channel='vx',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xy],
        raw_coords=raw_coords,
        channel='vy',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_xy],
        raw_coords=raw_coords,
        channel='vz',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.3 Y-Z Slices (at constant X)

In [ ]:
# === Parameters for Y-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_time_index_yz = 0
plot_x_index_yz = nx // 2
# ---
print(f"Using time-index {plot_time_index_yz} and x-index {plot_x_index_yz} for Y-Z slice plots.")

In [ ]:
# === Plot Y-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_yz],
        raw_coords=raw_coords,
        channel='vx',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_yz],
        raw_coords=raw_coords,
        channel='vy',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=timeseries_data[plot_time_index_yz],
        raw_coords=raw_coords,
        channel='vz',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

---

### 6. Time Evolution Plots (1D Line over Time)

#### 6.1 Evolution along Z-axis (at constant X, Y)

In [ ]:
# === Parameters for Time-Z Plots ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_tz = nx // 2
plot_y_index_tz = ny // 2
# ---
print(f"Using x-index {plot_x_index_tz} and y-index {plot_y_index_tz} for Time-Z plots.")

In [ ]:
# === Plot Time-Z for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_z_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vx',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Z for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_z_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vy',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Z for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_z_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vz',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 6.2 Evolution along Y-axis (at constant X, Z)

In [ ]:
# === Parameters for Time-Y Plots ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_ty = nx // 2
plot_z_index_ty = nz // 2
# ---
print(f"Using x-index {plot_x_index_ty} and z-index {plot_z_index_ty} for Time-Y plots.")

In [ ]:
# === Plot Time-Y for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_y_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vx',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Y for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_y_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vy',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Y for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_y_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vz',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 6.3 Evolution along X-axis (at constant Y, Z)

In [ ]:
# === Parameters for Time-X Plots ===
# <<< MODIFY THESE VALUES >>>
plot_y_index_tx = ny // 2
plot_z_index_tx = nz // 2
# ---
print(f"Using y-index {plot_y_index_tx} and z-index {plot_z_index_tx} for Time-X plots.")

In [ ]:
# === Plot Time-X for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_x_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vx',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-X for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_x_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vy',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-X for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_x_time_evolution(
        timeseries_data=timeseries_data,
        raw_coords=raw_coords,
        channel='vz',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )